In [ ]:
#!pip install nest_asyncio
#!pip install pyppeteer
#!pip install -U langgraph

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from IPython.display import display, Image
from langchain_core.runnables.graph import MermaidDrawMethod
import nest_asyncio

class onlineformstate(TypedDict):
    name: str
    mail_id: str
    number: str
    have_number: str
    alt_number: str
    submit: str

def get_name(state):
    state['name'] = input('Enter your name: ')
    return state

def get_mailid(state):
    state['mail_id'] = input('Enter mail id: ')
    return state

def get_number(state):
    state['number'] = input('Enter your phone number: ')
    return state

def validation(state):
    return state

def checklogic1(state):
    num = state['number']
    
    if len(num) == 10 and num.isdigit() and num[0] in ['6','7','8','9']:
        return 'validated'
    else:
        print('''\nInvalid number. Your number should be digits containing exactly 10 digits and should start either with 6/7/8/9. 
        Please re-enter a valid number''')
        return 'not validated'
        
def have_alt_number(state):
    state['have_number'] = input('Do you have alternate number? (yes/no): ')
    return state

def checklogic2(state): 
    if state['have_number'].lower() == 'yes':
        return 'yes' 
    else:
        return 'no'
        
def get_alternate_number(state):
    state['alt_number'] = input('Enter your alternate phone number: ')
    return state

def submit(state):
    state['submit'] = 'Done'
    return state

In [4]:
builder = StateGraph(onlineformstate)

builder.add_node('Enter name',get_name)
builder.add_node('Enter mail id',get_mailid)
builder.add_node('Enter phone number',get_number)
builder.add_node('Validation',validation)
builder.add_node('Have alternate number?',have_alt_number)
builder.add_node('Enter alternate number',get_alternate_number)
builder.add_node('Submit',submit)

builder.set_entry_point('Enter name')

builder.add_edge('Enter name','Enter mail id')
builder.add_edge('Enter mail id','Enter phone number')
builder.add_edge('Enter phone number','Validation')
builder.add_conditional_edges('Validation',checklogic1,{
    'validated':'Have alternate number?',
    'not validated':'Enter phone number'
})
builder.add_conditional_edges('Have alternate number?',checklogic2,{
    'yes':'Enter alternate number',
    'no':'Submit'
})
builder.add_edge('Enter alternate number','Submit')

builder.set_finish_point('Submit')

graph = builder.compile()

try:
    img = graph.get_graph().draw_mermaid_png(draw_method = MermaidDrawMethod.API, max_retries=5, retry_delay=2.0)
    display(Image(img))
    
except Exception as api_error:
    print('API visualization failed. Tryiing default method...')
    try:
        img = graph.get_graph().draw_mermaid_png(max_retries=5, retry_delay=2.0)
        display(Image(img))
    except Exception as default_error:
        print('Vizualization failed completely.')
        print('API Error: ',api_error)
        print('Default Error: ',default_error)
        print('Graph execution still works fine without vizualization')
        
#try:
    #display(Image(graph.get_graph().draw_mermaid_png()))
#except Exception as e:
    #print("Visualization failed. Ensure Mermaid & pygraphviz dependencies are installed.")
    #print(f"Error: {e}")

#nest_asyncio.apply()
#display(Image(graph.get_graph().draw_mermaid_png(draw_method=MermaidDrawMethod.PYPPETEER)))

API visualization failed. Tryiing default method...
Vizualization failed completely.
API Error:  Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 404.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
Default Error:  Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 404.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
Graph execution still works fine without vizualization


In [ ]:
result = graph.invoke({})
print(result)

In [ ]:
result = graph.invoke({})
print(result)